# HeartBeat (2013) green-plane pulse algorithm: modern port

This notebook runs the dependency-free Python port in `../heartbeat_port`. The port follows the original
`Algorithm.mm` and `Butterworth.cpp` (Or Maayan and Michael Leybovich, supervised by Aaron Wetzler). The original files in
`../../source/` are unchanged.

**All signals here are SYNTHETIC.** They exercise the code path and do not reproduce or validate the original
report's measurements. This student research prototype is **not a medical device**.

Run from `updated/notebooks/` (Jupyter, or `python3 ../tools/run_notebook.py heartbeat_port_demo.ipynb`).
In Colab, first clone the repository with its submodule and change into `updated/notebooks`.

In [ ]:
import sys, pathlib, math
# Use the local checkout, or fetch only the small updated Python modules in Colab.
cwd = pathlib.Path.cwd()
UPDATED_ROOT = next((p for d in [cwd, *cwd.parents] for p in (d, d / 'updated')
                     if (p / 'heartbeat_port' / '__init__.py').exists()), None)
if UPDATED_ROOT is None:
    import tempfile, urllib.request
    UPDATED_ROOT = pathlib.Path(tempfile.mkdtemp(prefix='heartbeat-updated-'))
    package = UPDATED_ROOT / 'heartbeat_port'
    package.mkdir()
    base = 'https://raw.githubusercontent.com/twerdster/heartbeat-ios/main/updated/heartbeat_port/'
    for name in ('__init__.py', 'algorithm.py', 'butterworth.py', 'frames.py', 'synthetic.py'):
        with urllib.request.urlopen(base + name, timeout=30) as response:
            (package / name).write_bytes(response.read())
sys.path.insert(0, str(UPDATED_ROOT))
from heartbeat_port import HeartbeatAlgorithm, butter, frequency_response, ORIGINAL_QUIRKS
from heartbeat_port.synthetic import synthetic_ppg_green_components
print('Updated Python modules:', UPDATED_ROOT)


The setup cell uses a local checkout when available. In a standalone notebook or Colab it downloads only the five small updated modules from this public repository. The optional original-code comparison runs only in a complete local Mac checkout. All signals below are synthetic.


## 1. The original band-pass filter
Order-5 Butterworth band-pass with cut-offs at 0.04 and 0.2 of Nyquist, which is 36 to 180 beats/min at 30 fps.

In [ ]:
b, a = butter((0.04, 0.2), 5)
print("b =", [f"{v:.6g}" for v in b])
print("a =", [f"{v:.6g}" for v in a])
for bpm in (20, 36, 60, 90, 120, 180, 300):
    w = math.pi * (bpm / 60) / 15  # rad/sample at 30 fps (Nyquist 15 Hz)
    print(f"{bpm:4d} bpm  |H| = {abs(frequency_response(b, a, w)):.4f}")

## 2. Running the original state machine on SYNTHETIC fingertip-like signals

In [ ]:
for true_bpm in (54, 84, 110, 140):
    alg = HeartbeatAlgorithm()
    first_final = None
    for k, c in enumerate(synthetic_ppg_green_components(true_bpm, 45, seed=true_bpm), start=1):
        alg.new_frame_with_green_component(c)
        if alg.is_final_result_determined() and first_final is None:
            first_final = (k / 30, alg.bpm_latest_result)
    print(f"SYNTHETIC {true_bpm:3d} bpm -> estimate after 45 s: {alg.bpm_latest_result:6.2f}; "
          f"first 'final' flag at {first_final[0]:.1f} s ({first_final[1]:.2f} bpm)")

## 3. Original behaviours that the port reproduces on purpose
The compiled original code (see `../harness/`) confirms each of these quirks.

In [ ]:
for q in ORIGINAL_QUIRKS:
    print("-", q)
alg = HeartbeatAlgorithm()
for _ in range(300):
    alg.new_frame_with_green_component(0.4)   # flat, pulseless input
print()
print("flat input -> peaks:", sum(alg.is_peak), "| reported bpm:", alg.bpm_latest_result,
      "| 'final':", alg.is_final_result_determined())

## 4. Optional: compare with the compiled original (macOS + Apple clang only)
This cell builds the unmodified original sources with a tiny UIColor shim and compares the two frame by frame.
It is skipped on other platforms.

In [ ]:
import shutil
if sys.platform == "darwin" and shutil.which("clang++") and (UPDATED_ROOT / "harness").exists():
    sys.path.insert(0, str(UPDATED_ROOT / "harness"))
    from compare_with_original import compare
    r = compare(synthetic_ppg_green_components(96, 40, seed=2), "strict")
    print(f"identical rows: {r['exact_rows']}/{r['frames']}")
else:
    print("skipped: needs macOS with Apple clang")

## Change the input and rerun
Edit `synthetic_bpm`, `seconds` and `seed` below to explore the signal and the algorithm response. The plot is synthetic test data, not a measurement of a person.


In [ ]:
synthetic_bpm, seconds, seed = 84, 30, 3
signal = synthetic_ppg_green_components(synthetic_bpm, seconds, seed=seed)
model = HeartbeatAlgorithm()
estimates = []
for component in signal:
    model.new_frame_with_green_component(component)
    estimates.append(model.bpm_latest_result)
from IPython.display import SVG, display
def trace(values, top, height, color):
    lo, hi = min(values), max(values)
    span = hi - lo or 1
    points = ' '.join(f'{40 + 720*i/max(1,len(values)-1):.2f},{top + height - height*(v-lo)/span:.2f}'
                      for i,v in enumerate(values))
    return f'<polyline fill="none" stroke="{color}" points="{points}"/>'
display(SVG('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 300">'
            '<rect width="800" height="300" fill="white"/>'
            f'<text x="40" y="20">SYNTHETIC {synthetic_bpm} bpm: green signal (top), estimate (bottom)</text>'
            + trace(signal, 35, 90, '#15803d') + trace(estimates, 160, 90, '#2563eb')
            + f'<text x="40" y="285">Final estimate: {model.bpm_latest_result:.2f} bpm; not a medical measurement</text></svg>'))
